# 9 Working with Polar Orbiting Sensor Datasets

## 9.1 Visible Infrared Imaging Radiometer Suite (VIIRS)


## 9.1.1 Importing and Plotting

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage as snd

from cartopy import crs as ccrs

import pvlib
import xarray as xr

In [ ]:
# Optional: Uniformly make font sizes larger and set default figure size
plt.rcParams.update({"font.size": 16, 
    "figure.figsize": [12, 12],
    "font.sans-serif": ["Segoe UI", "DejaVu Sans", "sans-serif"]
    })

In [ ]:
filename = "china_VJ202MOD.A2025352.0554.021.2025352130700.nc"
viirs_path = Path("./data/viirs") / filename
f1 = xr.open_datatree(viirs_path, mask_and_scale=False, \
                      engine="h5netcdf", decode_timedelta=True)
print("List of groups:", list(f1.groups))

In [ ]:
obs_group = f1["observation_data"]
print("Datasets in group:", list(obs_group))

In [ ]:
viirs_blue = obs_group["M03"]
viirs_blue.attrs

In [ ]:
sf_refl = viirs_blue.attrs["scale_factor"].item()
offset_refl = viirs_blue.attrs["add_offset"].item()

viirs_blue_refl = viirs_blue[:,:] * sf_refl + offset_refl

In [ ]:
sf_rad = viirs_blue.attrs["radiance_scale_factor"].item()
offset_rad = viirs_blue.attrs["radiance_add_offset"].item()

viirs_blue_rad = viirs_blue[:,:] * sf_rad + offset_rad

In [ ]:
CBAR_FRAC = 0.05
CBAR_PAD = 0.025
VMIN=0
VMAX=0.35
CMAP="Blues_r"

plt.figure(figsize=[10,5])
ax = plt.subplot()
img = ax.imshow(viirs_blue_refl, vmin=VMIN, vmax=VMAX, cmap=CMAP)
plt.colorbar(img, fraction=CBAR_FRAC, pad=CBAR_PAD, label="Reflectance")
ax.set(frame_on=True, xticks=[], yticks=[])
plt.show()

In [ ]:
filename = "china_VJ203MOD.A2025352.0554.021.2025352124629.nc"
viirs_path = Path("./data/viirs") / filename

f2 = xr.open_datatree(viirs_path, mask_and_scale=True, \
  engine="h5netcdf", decode_timedelta=True)

print("Groups:", list(f2.groups))

geo_group = f2["geolocation_data"]
print("Variables in geolocation_data:", list(geo_group))

In [ ]:
lats = geo_group["latitude"]
lons = geo_group["longitude"]

In [ ]:
fig = plt.figure(figsize=[10,5])
ax = plt.subplot(projection=ccrs.PlateCarree())
ax.coastlines("50m")

img = ax.pcolormesh(lons, lats, viirs_blue_refl, \
            vmin=VMIN, vmax=VMAX, cmap=CMAP)
plt.colorbar(img, fraction=CBAR_FRAC, pad=CBAR_PAD, 
            label="Reflectance")

plt.show()

## 9.1.2 Bowtie effects

In [ ]:
bowtie_flag = viirs_blue_refl.attrs["flag_values"][1]

In [ ]:
mask = (viirs_blue[:,:] == bowtie_flag)

In [ ]:
viirs_blue_refl = np.array(viirs_blue_refl)

In [ ]:
vza = geo_group["sensor_zenith"]
vaa = geo_group["sensor_azimuth"]

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=[10,10])

img = ax[0].imshow(viirs_blue_refl[0:300, :], \
                   vmin=VMIN, vmax=VMAX, cmap=CMAP)
fig.colorbar(img, orientation="horizontal")
ax[0].set_title("Reflectance")

img = ax[1].imshow(vza[0:300, :], cmap="jet")
fig.colorbar(img, orientation="horizontal", label="Degrees (°)")
ax[1].set_title("View Zenith Angle")

img = ax[2].imshow(vaa[0:300, :], cmap="jet")
fig.colorbar(img, orientation="horizontal", label="Degrees (°)")
ax[2].set_title("View Azimuth Angle")

for ax0 in ax:
  ax0.set(frame_on=True, xticks=[], yticks=[])

plt.tight_layout()
plt.show()

In [ ]:
viirs_blue_refl_cp = viirs_blue_refl.copy()
viirs_blue_refl_cp[mask] = 0

# Left side
left_boundary = np.where( (vza >= 31.72) & (vaa <= 0) )[1].max()
lhs = snd.maximum_filter1d( \
    viirs_blue_refl_cp[:, :left_boundary], \
    size=10, axis=0, mode="nearest")

# Right side
right_boundary = np.where( (vza >= 31.72) & (vaa >= 0) )[1].min()
rhs = snd.maximum_filter1d( \
    viirs_blue_refl_cp[:,right_boundary:], \
    size=10, axis=0, mode="nearest")

# Center unchanged
center = viirs_blue_refl_cp[:, left_boundary:right_boundary]

In [ ]:
viirs_blue_refl_nn = np.concatenate([lhs, center, rhs], axis=1)

In [ ]:
viirs_blue_refl_filled = np.where(mask, viirs_blue_refl_nn, viirs_blue_refl)

In [ ]:
fig = plt.figure(figsize=[10,5])

ax = plt.subplot(projection=ccrs.PlateCarree())
ax.coastlines("50m")

img = ax.pcolormesh(lons, lats, \
    viirs_blue_refl_filled, vmin=VMIN, vmax=VMAX, cmap=CMAP)
plt.colorbar(img, fraction=CBAR_FRAC, pad=CBAR_PAD)

plt.show()

In [ ]:
# Used for Fig 9.6
fig, ax = plt.subplots(1, 3, constrained_layout=True)

img = ax[0].imshow(viirs_blue_refl_cp[0:200, -200:], vmin=VMIN, vmax=VMAX, cmap="jet")
ax[0].set_title("(a) Uncorrected")

img = ax[1].imshow(viirs_blue_refl_nn[0:200, -200:], vmin=VMIN, vmax=VMAX, cmap="jet")
ax[1].set_title("(b) Nearest Neighbor")

img = ax[2].imshow(viirs_blue_refl_filled[0:200, -200:], vmin=VMIN, vmax=VMAX, cmap="jet")
fig.colorbar(img, orientation="vertical", fraction=0.05, pad=0.05)
ax[2].set_title("(c) Corrected")

for ax0 in ax.ravel():
  ax0.set(frame_on=True, xticks=[], yticks=[])

plt.show()

In [ ]:
start_y = 750
start_x = 1020
delta = 500

def create_sample( \
    var, start_x=4200, start_y=3300, \
    delta=500, step=1):
  end_x = start_x + delta
  end_y = start_y + delta
  return var[start_x:end_x, start_y:end_y][::step, ::step]

blue_sample = create_sample(np.flipud(np.fliplr(viirs_blue_refl_cp)), start_x=start_y, start_y=start_x, delta=delta, step=1)

In [ ]:
plt.figure(figsize=[8,8])
plt.imshow(blue_sample, cmap=CMAP, vmin=VMIN, vmax=0.2)
plt.colorbar(label="Reflectance", orientation="horizontal", fraction=0.046, pad=CBAR_PAD)
plt.gca().set_axis_off()

plt.tight_layout()
plt.show()

## 9.1.3 Unit Conversions

In [ ]:
am15 = pvlib.spectrum.get_reference_spectra(standard="ASTM G173-03")

In [ ]:
solar_irradiance = pvlib.spectrum.get_reference_spectra(standard="ASTM G173-03")
toa_irradiance_um = solar_irradiance["extraterrestrial"].reset_index()
toa_irradiance_um


In [ ]:
plt.figure(figsize=[10,5])

plt.plot(toa_irradiance_um["wavelength"], \
    toa_irradiance_um["extraterrestrial"])

plt.xlim(250, 1300)

plt.xlabel("Wavelength (nm)")
plt.ylabel("Irradiance (W m-2 nm-1)")
plt.title("Solar Spectral Irradiance (Top of Atmosphere)")

plt.show()

In [ ]:
def radiance_to_reflectance(L, Es):
    reflectance = np.pi * L / Es
    return reflectance

In [ ]:
solar_irradiance_490nm = \
    solar_irradiance[solar_irradiance.index == 490.0].extraterrestrial.item()
print(solar_irradiance_490nm)

In [ ]:
viirs_blue_refl_conv = radiance_to_reflectance(\
    viirs_blue_rad/1e3, solar_irradiance_490nm)

In [ ]:
def create_density(sample_orig, sample_corr, \
    label1="", label2=""):
  range = (0, 0.5)
  bins = (100, 100)
  title = f"{label1} vs. {label2}"

  density_orig, bins_orig = np.histogram(sample_orig, \
    bins=bins[0], density=True, range=range)
  density_corr, bins_corr = np.histogram(sample_corr, \
    bins=bins[1], density=True, range=range)

  plt.figure(figsize=[8,5])

  plt.plot(bins_orig[:-1], density_orig, label=label1)
  plt.plot(bins_corr[:-1], density_corr, label=label2)
  plt.legend()

  plt.xlim(range)

  plt.title(title)
  plt.xlabel("Normalized Reflectance")
  plt.ylabel("Probability Density")

  plt.show()

In [ ]:
create_density(viirs_blue_refl, viirs_blue_refl_conv, \
    label1="Unscaled Reflectance", label2="Converted from Radiance")

## 9.2 Atmospheric Correction

In [ ]:
sza = geo_group["solar_zenith"]
saa = geo_group["solar_azimuth"]

In [ ]:
label = "Degrees (°)"
proj = ccrs.PlateCarree()

fig, ax = plt.subplots(nrows=2, ncols=2, subplot_kw={'projection': proj})

ax[0,0].coastlines("50m")
img = ax[0,0].pcolormesh(lons, lats, sza, cmap="jet", transform=proj)
plt.colorbar(img, orientation="horizontal", label=label)
ax[0,0].set_title("Solar Zenith Angle")

ax[0,1].coastlines("50m")
img = ax[0,1].pcolormesh(lons, lats, saa, cmap="jet", transform=proj)
plt.colorbar(img, orientation="horizontal", label=label)
ax[0,1].set_title("Solar Azimuth Angle")

ax[1,0].coastlines("50m")
img = ax[1,0].pcolormesh(lons, lats, vza, cmap="jet", transform=proj)
plt.colorbar(img, orientation="horizontal", label=label)
ax[1,0].set_title("View Zenith Angle")

ax[1,1].coastlines("50m")
img = ax[1,1].pcolormesh(lons, lats, vaa, cmap="jet", transform=proj)
plt.colorbar(img, orientation="horizontal", label=label)
ax[1,1].set_title("View Azimuth Angle")

plt.show()

### 9.2.2 Dark Object Subtraction

In [ ]:
dark_pixel = viirs_blue_refl_filled.min()
print(dark_pixel)

In [ ]:
viirs_blue_refl_dos = viirs_blue_refl_filled - dark_pixel

In [ ]:
create_density(viirs_blue_refl_filled, viirs_blue_refl_dos, \
    label1="Original", label2="Dark Object Subtraction")

### 9.2.2 Rayleigh Correction

In [ ]:
def rayleigh_single(xtau, xphi, xmus, xmuv):
  # Kronecker Terms (Eq. 9.4)
  kron = [1.0, 2.0, 2.0]

  # Cosine terms (Eq. 9.4 )
  phios = np.pi - xphi
  cosf = [1.0, np.cos(phios), np.cos(2.0*phios)]

  # Rayleigh phase function terms (Eq. 9.8-9.9)
  beta = 0.5
  gamma = 0.0279

  # Constant from Eqn. 9.10, used in Eqn. 9.7-9.9
  alpha = (1-gamma/(2-gamma))/(1+2*gamma/(2-gamma))

  # Eqs. 9.7-9.9
  P0 = 1.0 + (3*xmus**2-1.0)*(3*xmuv**2-1)*alpha/8.0
  P1 = -xmus*xmuv*np.sqrt(1-xmus**2) \
      * np.sqrt(1-xmuv**2) \
      * alpha*beta*1.5
  P2 = (1-xmus**2)*(1-xmuv**2)*alpha*beta*0.375

  # Eq. 9.6
  xitm = (1-np.exp(-xtau*(1/xmus+1/xmuv)))*xmus/(4*(xmus+xmuv))
  phase = [P0*xitm, P1*xitm, P2*xitm]

  # Single scattering component (Eq. 9.4)
  xray = kron[0]*phase[0]*cosf[0] \
    + kron[1]*phase[1]*cosf[1] \
    + kron[2]*phase[2]*cosf[2]
  
  # Normalize by the sza
  xray = xray/xmus

  return xray

In [ ]:
xtau = 0.13379

In [ ]:
solar_zenith_rad = np.deg2rad(sza)
view_zenith_rad = np.deg2rad(vza)
relative_azimuth_deg = np.abs(saa - vaa)
relative_azimuth_deg = np.where(relative_azimuth_deg > 180, 
                                360 - relative_azimuth_deg, 
                                relative_azimuth_deg)
relative_azimuth_rad = np.deg2rad(relative_azimuth_deg)

In [ ]:
# Rayleigh scattering function
# xtau: raylegh optical depth 
# xphi: azimuthal difference between sun and observation (xphi=0, in backscattering) and expressed in degree (0 to 360.)
# xmus: cosine of the solar zenith angle
# xmuv: cosine of the observation/view zenith angle

xphi = relative_azimuth_rad
xmus = np.cos(solar_zenith_rad)
xmuv = np.cos(view_zenith_rad)

In [ ]:
ray_blue_refl = rayleigh_single(xtau, xphi, xmus, xmuv)

In [ ]:
fig = plt.figure(figsize=[10,5])
ax = plt.subplot(projection=proj)
ax.coastlines("50m")

img = ax.pcolormesh(lons, lats, ray_blue_refl, \
    cmap="jet", vmin=0.0, transform=proj)
fig.colorbar(img, orientation="vertical", pad=0.01, shrink=0.8)

ax.set_title("Rayleigh Reflectance")

ax.set(frame_on=True, xticks=[], yticks=[])

plt.tight_layout()
plt.show()

In [ ]:
create_density(viirs_blue_refl, viirs_blue_refl_filled-ray_blue_refl, \
    label1="Original", label2="Rayleigh-Corrected")